# SepsisGuard — GRPO Training Notebook

Train 4 multi-agent roles (Nurse, Lab, Pharmacist, Physician) using TRL GRPO with Unsloth 4-bit quantization.

This notebook:
1. Loads a quantized Qwen 2.5-3B model
2. Connects to the live SepsisGuard environment
3. Collects initial rollouts for the prompt dataset
4. Runs GRPO training with online environment rewards
5. Evaluates and plots reward improvement vs heuristic baseline

In [ ]:
!pip install -q -U "unsloth[colab-new]" openenv-core "trl>=0.12" vllm datasets matplotlib
!pip install -q requests httpx

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=4096, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)
FastLanguageModel.for_inference(model)
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
from unsloth import FastLanguageModel
import torch

# --- 1. Define the path where you saved the model ---
# We will use the LoRA path, as it is extremely fast to load 
# and automatically pairs with the base model you used.
saved_model_path = "/data/sepsis-model/sepsis-grpo-lora"

# Use the same sequence length you used during training
max_seq_length = 3000 

print(f"Loading trained model from {saved_model_path}...")

# --- 2. Load the Model and Tokenizer ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = saved_model_path,
    max_seq_length = max_seq_length,
    dtype = None, # Auto-detects fp16/bf16 based on your GPU
    load_in_4bit = False, # Set to True if you are critically low on VRAM
)

# --- 3. Switch to Inference Mode ---
# This optimizes the model for generation/evaluation rather than training
FastLanguageModel.for_inference(model)

print("✅ Model successfully loaded and ready for evaluation!")

In [ ]:
%cd /data/sepsisguard

In [ ]:
import os
import sys

# Define a relative folder name instead of an absolute Colab path
repo_folder = "sepsisguardv2"

# 1. Clone if it doesn't exist in the current directory
if not os.path.exists(repo_folder):
    !git clone -b sepsisguard-v2 https://github.com/JishnuVijayan/Sepsis-Guard.git sepsisguardv2

# 2. Change into the newly cloned directory
os.chdir(repo_folder)

# 3. Add the current directory (which is now sepsisguard) to sys.path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# 4. Install the package
!pip install -q -e .

# 5. Verify
print("Repo ready:", os.listdir("."))

In [ ]:
!pip install -q -e .

In [ ]:
%pwd

In [ ]:
import os
import signal
import subprocess
import time
import requests
import sys

# Dynamically get the current directory (should be your sepsisguard folder)
REPO_DIR = os.getcwd()
LOG_PATH = os.path.join(REPO_DIR, "uvicorn.log")

PORT = 8000
UVICORN_CMD = f"uvicorn server.app:app --host 0.0.0.0 --port {PORT}"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def _find_server_pids():
    proc = subprocess.run(
        ["pgrep", "-f", f"uvicorn server.app:app.*--port {PORT}"],
        capture_output=True, text=True
    )
    if proc.returncode != 0 or not proc.stdout.strip():
        return []
    return [int(x) for x in proc.stdout.strip().splitlines() if x.strip().isdigit()]

def _stop_existing_server():
    pids = _find_server_pids()
    if not pids:
        print("No existing SepsisGuard uvicorn process found on port", PORT)
        return
    print(f"Stopping existing uvicorn process(es): {pids}")
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    time.sleep(1.5)
    
    survivors = _find_server_pids()
    for pid in survivors:
        try:
            os.kill(pid, signal.SIGKILL)
        except OSError:
            pass

def _start_server():
    # Write the log file to the dynamic current directory
    log_file = open(LOG_PATH, "w")
    
    subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", str(PORT)],
        stdout=log_file,
        stderr=log_file,
        cwd=REPO_DIR # Start the server in the dynamic current directory
    )

def _wait_for_health(timeout_s=15):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            r = requests.get(HEALTH_URL, timeout=3)
            if r.ok:
                return True
        except Exception:
            pass
        time.sleep(1)
    return False

_stop_existing_server()
print("Starting server...")
_start_server()

if _wait_for_health(timeout_s=15):
    print(f"✅ Local SepsisGuard server is healthy at http://127.0.0.1:{PORT}")
    print(f"📄 Log file created at: {LOG_PATH}")
else:
    print("❌ Server started but health check failed. Here is the crash log:")
    print("-" * 40)
    with open(LOG_PATH, "r") as f:
        print(f.read())
    print("-" * 40)

In [ ]:
!curl "http://127.0.0.1:8000/health"

In [ ]:
import os, requests, time

# If notebook and server run on the SAME Colab VM, localhost works.
# If your server is outside Colab, set ENV_BASE_URL to a public tunnel URL.
ENV_URL = os.environ.get("ENV_BASE_URL", "http://127.0.0.1:8000")

class EnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")

    def reset(self, task_name, seed, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/reset",
                          json={"task_name": task_name, "seed": seed},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def step(self, actions, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/step",
                          json={"actions": actions},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def create_session(self):
        r = requests.post(f"{self.base_url}/session", timeout=10)
        r.raise_for_status()
        return r.json()["session_id"]

    def delete_session(self, session_id):
        try:
            requests.delete(f"{self.base_url}/session/{session_id}", timeout=5)
        except Exception:
            pass

# Wait briefly for server readiness so first /reset doesn't fail noisily.
ready = False
for _ in range(15):
    try:
        requests.get(f"{ENV_URL}/state", timeout=3).raise_for_status()
        ready = True
        break
    except Exception:
        time.sleep(1)
if not ready:
    raise RuntimeError(
        f"Server not reachable at {ENV_URL}. Start uvicorn or set ENV_BASE_URL to your reachable URL."
    )

env = EnvClient(ENV_URL)
info = env.reset(task_name="task1_textbook", seed=42)
print(f"Connected to {ENV_URL}")
print(f"Tick: {info['info']['tick']}, Roles: {list(info['observations'].keys())}")

In [ ]:
import sys
from collections import Counter
sys.path.insert(0, "/data/sepsisguardv2")

from agents.nurse import HeuristicNurse
from agents.lab import HeuristicLab
from agents.pharmacist import HeuristicPharmacist
from agents.physician import HeuristicPhysician
from training.rollout_collector import collect_prompt_rollouts

ROLES = ("nurse", "lab", "pharmacist", "physician")
N_EPISODES = 60
SEEDS = list(range(42, 42 + N_EPISODES))
TASK = "task1_textbook"
MAX_TICKS_PER_EP = 48
INCLUDE_QUIET = True

heuristic_agents = {
    "nurse": HeuristicNurse(),
    "lab": HeuristicLab(),
    "pharmacist": HeuristicPharmacist(),
    "physician": HeuristicPhysician(),
}

print(f"Collecting prompts from {N_EPISODES} heuristic episodes...")
rollouts = collect_prompt_rollouts(
    env_client=env,
    heuristic_agents=heuristic_agents,
    task_name=TASK,
    seeds=SEEDS,
    max_ticks_per_episode=MAX_TICKS_PER_EP,
    include_quiet=INCLUDE_QUIET,
)

role_counts = Counter(r["role"] for r in rollouts)
actionable_counts = Counter(r["role"] for r in rollouts if r["actionable_signal"])

print(f"\nCollected {len(rollouts)} prompt rows")
print(f"Role counts: {dict(role_counts)}")
print(f"Actionable counts: {dict(actionable_counts)}")
print(f"Quiet share: {1 - (sum(actionable_counts.values()) / max(1, len(rollouts))):.2%}")



In [ ]:
!curl "http://127.0.0.1:8000/health"

In [ ]:
from collections import Counter
from training.rollout_collector import dedupe_rollouts, build_prompt_dataset

original_count = len(rollouts)
deduped_rollouts = dedupe_rollouts(rollouts, keep_actionable_duplicates=2)
train_dataset = build_prompt_dataset(
    deduped_rollouts,
    dedupe=False,
    quiet_keep_ratio=0.40,
)

role_counts = Counter(r["role"] for r in deduped_rollouts)
actionable_counts = Counter(r["role"] for r in deduped_rollouts if r["actionable_signal"])

print(f"Original rollout rows: {original_count}")
print(f"After clinical dedupe: {len(deduped_rollouts)}")
print(f"Final train prompts: {len(train_dataset)}")
print(f"Role counts after dedupe: {dict(role_counts)}")
print(f"Actionable counts after dedupe: {dict(actionable_counts)}")



In [ ]:
import json
import re
import requests
import torch
from training.prompts import build_role_prompt
from agents.nurse import HeuristicNurse
from agents.lab import HeuristicLab
from agents.pharmacist import HeuristicPharmacist
from agents.physician import HeuristicPhysician

PRECHECK_EVAL_EPISODES = 2
PRECHECK_EVAL_SEEDS = list(range(100, 100 + PRECHECK_EVAL_EPISODES))

def _precheck_make_llm_agent_fn(model, tokenizer, target_role):
    fallback_agents = {
        'nurse': HeuristicNurse(),
        'lab': HeuristicLab(),
        'pharmacist': HeuristicPharmacist(),
        'physician': HeuristicPhysician(),
    }
    def agent_fn(role, obs):
        if role != target_role:
            return fallback_agents[role].decide(obs)
        prompt = build_role_prompt(obs, role)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=192, do_sample=False)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        try:
            parsed = json.loads(text.strip())
            if isinstance(parsed, dict) and 'operation' in parsed:
                return parsed
        except Exception:
            pass
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                if isinstance(parsed, dict) and 'operation' in parsed:
                    return parsed
            except Exception:
                pass
        return fallback_agents[role].decide(obs)
    return agent_fn

def _precheck_run_episode(env_client, task_name, seed, agent_fn, session_id=None):
    bundle = env_client.reset(task_name=task_name, seed=seed, session_id=session_id)
    done = False
    while not done:
        obs = bundle['observations']
        actions = {role: agent_fn(role, obs[role]) for role in ('nurse', 'lab', 'pharmacist', 'physician')}
        bundle = env_client.step(actions, session_id=session_id)
        done = bundle['done']
    grader = requests.get(
        f'{env_client.base_url}/grader',
        headers={'X-Session-Id': session_id} if session_id else {},
        timeout=30,
    ).json()
    return grader.get('score', 0.0)

FastLanguageModel.for_inference(model)
precheck_scores = {}
for target_role in ('nurse', 'lab', 'pharmacist', 'physician'):
    llm_fn = _precheck_make_llm_agent_fn(model, tokenizer, target_role)
    scores = []
    for seed in PRECHECK_EVAL_SEEDS:
        sid = env.create_session()
        score = _precheck_run_episode(env, 'task1_textbook', seed, llm_fn, session_id=sid)
        scores.append(score)
        env.delete_session(sid)
    precheck_scores[target_role] = scores

print('Live pre-validation check (2 episodes each):')
for role, scores in precheck_scores.items():
    mean_score = sum(scores) / len(scores)
    print(f'{role:<12} mean={mean_score:.4f} scores={[round(s, 3) for s in scores]}')


In [ ]:
# =====================================================================
# BYPASS PRE-EVALUATION: HARDCODED BASELINES
# =====================================================================

# 1. The seeds used for evaluation
N_EVAL = 2
EVAL_SEEDS = list(range(100, 100 + N_EVAL))

# 2. Heuristic baseline scores (2-episode subset)
pre_baseline_scores = [0.938, 0.95]

# 3. Pre-train LLM scores for each role (2-episode subset)
pre_trained_scores = {
    "nurse":      [0.938, 0.95],
    "lab":        [0.7, 0.275],
    "pharmacist": [0.738, 0.95],
    "physician":  [0.938, 0.95],
}

print("Pre-evaluation baselines loaded (2-episode subset).")



In [ ]:
import torch
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback
from training.reward_shaping import make_online_sepsis_reward_fn, format_reward_fn

FastLanguageModel.for_training(model)

has_cuda = torch.cuda.is_available()
if has_cuda:
    major, _ = torch.cuda.get_device_capability()
    use_bf16 = major >= 8
    use_fp16 = not use_bf16
    print(f"CUDA device: {torch.cuda.get_device_name(0)} | bf16={use_bf16} fp16={use_fp16}")
else:
    use_bf16 = False
    use_fp16 = False
    print("CUDA not available. Using fp32 precision.")

cfg = GRPOConfig(
    output_dir="./sepsis-grpo-v4",
    num_generations=8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=450,
    learning_rate=2e-5,
    warmup_steps=40,
    logging_steps=5,
    save_steps=150,
    max_prompt_length=3800,
    max_completion_length=160,
    temperature=1.0,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    gradient_checkpointing=True,
)

reward_fn_env_obj = make_online_sepsis_reward_fn(
    env_url=ENV_URL,
    task_name=TASK,
    seed=42,
    warmup_ticks=6,
    inject_ticks=2,
    max_workers=3,
    verbose=False,
)

preflight_ok = reward_fn_env_obj.preflight_check()
if not preflight_ok:
    print('Warning: reward preflight did not find a discriminative case. Continuing, but check server/task alignment if rewards look flat.')

def reward_fn_env(*args, **kwargs):
    return reward_fn_env_obj(*args, **kwargs)
reward_fn_env.__name__ = "reward_fn_env"

reward_log = {"steps": [], "env_reward": [], "format_reward": [], "combined_reward": []}

class RewardLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        env_r = logs.get("rewards/reward_fn_env/mean") or logs.get("reward_fn_env", 0.0)
        fmt_r = logs.get("rewards/format_reward_fn/mean") or logs.get("format_reward_fn", 0.0)
        combined = logs.get("reward", env_r + fmt_r)
        if env_r != 0.0 or fmt_r != 0.0 or "reward" in logs:
            reward_log["steps"].append(step)
            reward_log["env_reward"].append(float(env_r))
            reward_log["format_reward"].append(float(fmt_r))
            reward_log["combined_reward"].append(float(combined))

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn_env, format_reward_fn],
    args=cfg,
    train_dataset=train_dataset,
    callbacks=[RewardLogger()],
)

print(f"Training: {cfg.max_steps} steps, lr={cfg.learning_rate}, batch={cfg.per_device_train_batch_size}x{cfg.gradient_accumulation_steps}, gen={cfg.num_generations}")
print(f"Dataset prompts: {len(train_dataset)}")
trainer.train()




In [ ]:
import os

# --- 1. Define Paths ---
# Persistent Storage Paths
base_dir = "/data/sepsis-model"
lora_path = os.path.join(base_dir, "sepsis-grpo-v4-lora")       # NEW: Added -v2
merged_path = os.path.join(base_dir, "sepsis-grpo-v4-merged")   # NEW: Added -v2

# Ensure the base directory exists
os.makedirs(base_dir, exist_ok=True)

# --- 2. Save Directly to Persistent Storage ---
print(f"💾 Saving V4 LoRA adapters to {lora_path} ...")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

print(f"💾 Saving V4 Merged 16-bit model to {merged_path} ...")
print("(This may take a few minutes...))")
model.save_pretrained_merged(
    merged_path,
    tokenizer,
    save_method="merged_16bit",
)

print(f"\n🎉 Success! V3 Models are securely saved to your persistent storage at {base_dir}.")

In [ ]:
def make_llm_agent_fn(model, tokenizer, target_role):
    def agent_fn(role, obs):
        if role != target_role:
            return heuristic_agents_eval[role].decide(obs)
        prompt = build_role_prompt(obs, role)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            parsed = json.loads(text.strip())
            if isinstance(parsed, dict) and "operation" in parsed: return parsed
        except Exception: pass
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                if isinstance(parsed, dict) and "operation" in parsed: return parsed
            except Exception: pass
        return heuristic_agents_eval[role].decide(obs)
    return agent_fn

def run_episode(env_client, task_name, seed, agent_fn, session_id=None):
    bundle = env_client.reset(task_name=task_name, seed=seed, session_id=session_id)
    done = False
    while not done:
        obs = bundle["observations"]
        actions = {role: agent_fn(role, obs[role]) for role in ("nurse", "lab", "pharmacist", "physician")}
        bundle = env_client.step(actions, session_id=session_id)
        done = bundle["done"]
    grader = requests.get(f"{env_client.base_url}/grader",
                          headers={"X-Session-Id": session_id} if session_id else {},
                          timeout=30).json()
    return grader.get("score", 0.0)

nurse_h, lab_h, pharma_h, phys_h = HeuristicNurse(), HeuristicLab(), HeuristicPharmacist(), HeuristicPhysician()
def heuristic_agent_fn(role, obs):
    return heuristic_agents_eval[role].decide(obs)
heuristic_agents_eval = {"nurse": nurse_h, "lab": lab_h, "pharmacist": pharma_h, "physician": phys_h}

In [ ]:
import requests, json, re, torch

FastLanguageModel.for_inference(model)

post_trained_scores = {}
for target_role in ("nurse", "lab", "pharmacist", "physician"):
    llm_fn = make_llm_agent_fn(model, tokenizer, target_role)
    scores = []
    for seed in EVAL_SEEDS:
        sid = env.create_session()
        score = run_episode(env, "task1_textbook", seed, llm_fn, session_id=sid)
        scores.append(score)
        env.delete_session(sid)
    post_trained_scores[target_role] = scores
    pre_mean = sum(pre_trained_scores[target_role]) / len(pre_trained_scores[target_role])
    post_mean = sum(scores) / len(scores)
    delta = post_mean - pre_mean
    print(f"post[{target_role}] mean={post_mean:.4f} delta_vs_pre={delta:+.4f} scores={[round(s, 3) for s in scores]}")

heuristic_mean = sum(pre_baseline_scores) / len(pre_baseline_scores)
print(f"\n{'Role':<15} {'Heuristic':>10} {'Pre-Train':>10} {'Post-Train':>10} {'Delta':>10}")
print("-" * 60)
for role in ("nurse", "lab", "pharmacist", "physician"):
    pre = sum(pre_trained_scores[role]) / len(pre_trained_scores[role])
    post = sum(post_trained_scores[role]) / len(post_trained_scores[role])
    delta = post - pre
    print(f"{role:<15} {heuristic_mean:>10.4f} {pre:>10.4f} {post:>10.4f} {delta:>+10.4f}")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- SAFETY CATCH FOR MISSING VARIABLES ---
# If the kernel restarted, reward_log is gone. Let's gracefully handle it.
try:
    reward_log
except NameError:
    print("⚠️ Warning: 'reward_log' not found in memory. Skipping Panel 1 data.")
    reward_log = {"steps": [], "env_reward": [], "format_reward": [], "combined_reward": []}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# --- Panel 1: Reward Curves ---
if reward_log.get("steps"):
    axes[0].plot(reward_log["steps"], reward_log["env_reward"], label="Env Reward", linewidth=1.5, alpha=0.8)
    axes[0].plot(reward_log["steps"], reward_log["format_reward"], label="Format Reward", linewidth=1.5, alpha=0.8)
    if reward_log.get("combined_reward"):
        axes[0].plot(reward_log["steps"], reward_log["combined_reward"], label="Combined", linewidth=2, color="black", alpha=0.5)
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Mean Reward")
    axes[0].set_title("GRPO Training Reward Curves")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No reward logs captured\n(kernel restarted since training)",
                 ha="center", va="center", transform=axes[0].transAxes, fontsize=11)
    axes[0].set_title("GRPO Training Reward Curves")

# --- Panel 2: Before vs After (per role) ---
roles = ["nurse", "lab", "pharmacist", "physician"]
heuristic_mean = sum(pre_baseline_scores) / len(pre_baseline_scores)

pre_means = [sum(pre_trained_scores[r]) / len(pre_trained_scores[r]) for r in roles]
post_means = [sum(post_trained_scores[r]) / len(post_trained_scores[r]) for r in roles]

x = np.arange(len(roles))
width = 0.35
bars_pre = axes[1].bar(x - width/2, pre_means, width, label="Pre-Training", color="#FF9800", alpha=0.8)
bars_post = axes[1].bar(x + width/2, post_means, width, label="Post-Training", color="#4CAF50", alpha=0.8)
axes[1].axhline(y=heuristic_mean, color="#888888", linestyle="--", alpha=0.7, label=f"Heuristic ({heuristic_mean:.3f})")
axes[1].set_xticks(x)
axes[1].set_xticklabels([r.capitalize() for r in roles])
axes[1].set_ylabel("Episode Score")
axes[1].set_title("Pre-Training vs Post-Training (per role)")
axes[1].set_ylim(0, 1.0)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")

for bar, val in zip(bars_pre, pre_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
                 ha="center", va="bottom", fontsize=7)
for bar, val in zip(bars_post, post_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
                 ha="center", va="bottom", fontsize=7)

# --- Panel 3: Delta improvement ---
deltas = [post - pre for post, pre in zip(post_means, pre_means)]
colors = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
bars_d = axes[2].bar(roles, deltas, color=colors, alpha=0.8)
axes[2].axhline(y=0, color="gray", linestyle="-", alpha=0.5)
axes[2].set_ylabel("Score Change (Post - Pre)")
axes[2].set_title("Training Improvement by Role")
axes[2].grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars_d, deltas):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 val + 0.01 if val >= 0 else val - 0.03,
                 f"{val:+.3f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=9)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to training_results.png")

In [ ]:
import json
import matplotlib.pyplot as plt
import os

# --- 1. Point to your highest checkpoint folder ---
# Since you trained for 1000 steps with save_steps=250, checkpoint-1000 should exist.
# (If you stopped early, change this to checkpoint-750, etc.)
checkpoint_dir = "./sepsisguard/sepsis-grpo-v2/checkpoint-200" 
state_path = os.path.join(checkpoint_dir, "trainer_state.json")

print(f"🔍 Attempting to recover logs from: {state_path}")

try:
    with open(state_path, "r") as f:
        state = json.load(f)
    
    steps = []
    env_rewards = []
    format_rewards = []
    combined_rewards = []
    
    # --- 2. Extract metrics from Hugging Face's log history ---
    for log in state.get("log_history", []):
        # TRL logs the rewards under these specific keys
        env_r = log.get("rewards/reward_fn_env/mean")
        if env_r is None:
            env_r = log.get("reward_fn_env")
            
        fmt_r = log.get("rewards/format_reward_fn/mean")
        if fmt_r is None:
            fmt_r = log.get("format_reward_fn")
            
        if env_r is not None and fmt_r is not None:
            steps.append(log.get("step"))
            env_rewards.append(float(env_r))
            format_rewards.append(float(fmt_r))
            combined_rewards.append(float(env_r) + float(fmt_r))
            
    print(f"✅ Successfully recovered {len(steps)} log entries!")

    # --- 3. Plot the Reward Curves ---
    if steps:
        plt.figure(figsize=(10, 6))
        plt.plot(steps, env_rewards, label="Environment Reward", linewidth=1.5, alpha=0.8, color="#1f77b4")
        plt.plot(steps, format_rewards, label="Format Reward", linewidth=1.5, alpha=0.8, color="#ff7f0e")
        plt.plot(steps, combined_rewards, label="Total Combined Reward", linewidth=2.5, color="black", alpha=0.7)

        plt.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
        plt.xlabel("Training Step", fontsize=12)
        plt.ylabel("Mean Reward", fontsize=12)
        plt.title("GRPO Training Reward Convergence v1", fontsize=14, fontweight="bold")
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig("reward_curves_recovered.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("🎉 Plot saved to reward_curves_recovered.png")
    else:
        print("⚠️ Could not find reward metrics in the log history.")

except FileNotFoundError:
    print(f"❌ Error: Could not find {state_path}. Look inside your './sepsis-grpo' folder and update the 'checkpoint_dir' variable to match the highest checkpoint folder there!")

In [ ]:
import os
import zipfile

# --- 1. Define Paths ---
base_dir = "/data/sepsis-model"
lora_dir = os.path.join(base_dir, "sepsis-grpo-lora")
merged_dir = os.path.join(base_dir, "sepsis-grpo-merged")
zip_output_path = os.path.join(base_dir, "sepsis-models-backup.zip")

print(f"📦 Starting zip process... Saving to: {zip_output_path}")

# --- 2. Create the Zip Archive ---
with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    
    # Helper function to add a directory to the zip
    def add_dir(target_dir):
        if not os.path.exists(target_dir):
            print(f"⚠️ Warning: Directory not found -> {target_dir}")
            return
            
        for root, dirs, files in os.walk(target_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # This keeps the folder structure clean inside the zip
                arcname = os.path.relpath(file_path, base_dir)
                zipf.write(file_path, arcname)
                
    print("Adding LoRA adapters...")
    add_dir(lora_dir)
    
    print("Adding Merged Model (This might take a moment depending on size)...")
    add_dir(merged_dir)

print(f"✅ Zip complete! File is ready at: {zip_output_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

roles = ["nurse", "lab", "pharmacist", "physician"]
x = np.arange(len(roles))
width = 0.25

# Scores from your previous V1 run (from the screenshot)
v1_means = [0.8425, 0.4800, 0.7250, 0.8125] 

# Scores from this current V2 run
v2_means = [0.7575, 0.4250, 0.7625, 0.4500]

# Pre-train/Base scores from your log
base_means = [0.7526, 0.3998, 0.6700, 0.7552]

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(x - width, base_means, width, label='Base (Initial)', color='#9E9E9E')
ax.bar(x, v1_means, width, label='V1 (200 steps - PEAK)', color='#2196F3')
ax.bar(x + width, v2_means, width, label='V2 (800 steps - COLLAPSE)', color='#f44336')

ax.set_ylabel('Mean Score')
ax.set_title('SepsisGuard Performance: The V2 Regression')
ax.set_xticks(x)
ax.set_xticklabels([r.capitalize() for r in roles])
ax.legend()

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()